# Chạy thử crawler trên một khoảng ID (code mới, chưa push) — có proxy và WARP

Notebook này chạy **thật** (gửi request tới arXiv và Semantic Scholar) nhưng chỉ trên khoảng ID bạn chọn,
với đủ 5 file mới trong `D:\NMKHDL\gop-y-member1\src`. Dữ liệu ghi vào `02_trial`, không đụng `01_crawl`.

- Chạy **lần lượt từ trên xuống** (Shift + Enter từng ô).
- Notebook này **bật xoay proxy Webshare** (secret `ARXIV_PROXIES`) cho các lệnh gọi API arXiv và **bật WARP**
  cho việc tải file (xen kẽ mạng Colab ↔ WARP). Tắt từng cái ở Bước 3b.
  ⚠️ FAQ của Colab cấm *"connecting to remote proxies"*: Colab có thể ngắt runtime giữa chừng. Nếu bị ngắt bất thường,
  đặt hai công tắc ở Bước 3b thành `False` rồi chạy lại; dữ liệu đã xong không mất.
- Xoay proxy/WARP **không** làm crawl nhanh hơn: mọi request (qua IP nào cũng vậy) đi chung một bộ giới hạn.
- Nên mở **runtime mới** trước khi chạy (*Runtime → Disconnect and delete runtime*).
- Secret `S2_API_KEY` (key riêng của bạn) và `ARXIV_PROXIES` dùng lại cái bạn đã tạo, nhớ bật *Notebook access*
  cho notebook này. **Không** dán key hay mật khẩu vào ô code nào.

## Bước 1. Kết nối Google Drive
Chọn đúng tài khoản có thư mục `group11-data` → *Allow*.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Bước 2. Chọn khoảng ID
Chỉ sửa `TU` và `DEN` (tính cả hai đầu, phải nằm trong khoảng được giao 2605.12338 .. 2605.17337).
Mỗi khoảng có thư mục riêng `02_trial/m1_<đầu>_<cuối>`; chạy lại cùng khoảng thì crawler làm tiếp phần còn thiếu.
Khoảng 100 ID mất chừng 5–7 phút.

In [ ]:
# ===== Chỉ sửa ở đây =====
TU  = "2605.15000"   # ID đầu
DEN = "2605.15099"   # ID cuối
NHOM_DIR = "/content/drive/MyDrive/group11-data"
BRANCH = "quan-download-robust"
# =========================
import os, re
GIAO = (12338, 17337)  # khoảng được giao của Member 1
k1, k2 = re.fullmatch(r"2605\.(\d{5})", TU.strip()), re.fullmatch(r"2605\.(\d{5})", DEN.strip())
if not (k1 and k2):
    raise SystemExit("TU và DEN phải có dạng 2605.xxxxx (5 chữ số sau dấu chấm)")
a, b = int(k1[1]), int(k2[1])
if not GIAO[0] <= a <= b <= GIAO[1]:
    raise SystemExit(f"Cần 2605.{GIAO[0]} <= TU <= DEN <= 2605.{GIAO[1]}")
if not os.path.isdir(NHOM_DIR):
    raise SystemExit(f"Không thấy {NHOM_DIR}. Nếu thư mục do người khác chia sẻ: chuột phải -> "
                     "'Thêm lối tắt vào Drive' rồi chạy lại ô này.")
TU, DEN, SO_ID = f"2605.{a}", f"2605.{b}", b - a + 1
THU = f"{NHOM_DIR}/02_trial/m1_{a}_{b}"
os.makedirs(THU, exist_ok=True)
DS_ID = f"{THU}/ds_id.txt"  # danh sách ID của khoảng, để validate.py kiểm tra không sót ID nào
with open(DS_ID, "w", encoding="utf-8") as f:
    f.write("\n".join(f"2605.{n}" for n in range(a, b + 1)) + "\n")
print(f"{SO_ID} ID: {TU} .. {DEN}")
print("Ghi vào:", THU)
print("Đã có dữ liệu từ lần trước:", os.path.isfile(f"{THU}/work/paper_manifest.jsonl"))

## Bước 3. Lấy code và thay 5 file mới
Ô 1 lấy code của nhóm trên GitHub (và trả mọi file về bản GitHub, nên chạy lại bao nhiêu lần cũng được).
Ô 2 hỏi file: chọn **cùng lúc 5 file** trong `D:\NMKHDL\gop-y-member1\src` (giữ Ctrl khi bấm):
`rate_limiter.py`, `downloader.py`, `main.py`, `group_benchmark.py`, `ref_dates_kaggle.py`.
(Đừng chọn `proxies.txt`: proxy lấy từ Secrets.) Mỗi runtime mới phải tải lại 5 file này (đến khi bạn push).

In [ ]:
REPO = "/content/group11-arxiv-project"
os.chdir("/content")
if os.path.isdir(f"{REPO}/.git"):
    !git -C {REPO} checkout -q -- .
    !git -C {REPO} pull -q
else:
    !git clone -q -b {BRANCH} https://github.com/mituleefam/group11-arxiv-project.git {REPO}
os.chdir(REPO)
!git log --oneline -1
!pip install -q -r src/requirements.txt

In [ ]:
import shutil
from google.colab import files
CAN = ["rate_limiter.py", "downloader.py", "main.py", "group_benchmark.py", "ref_dates_kaggle.py"]
TAM = "/content/tai_len"
shutil.rmtree(TAM, ignore_errors=True)
os.makedirs(TAM)
os.chdir(TAM)  # files.upload() lưu thêm một bản ở thư mục hiện tại: để nó ở đây, không lẫn vào repo
try:
    up = files.upload()
finally:
    os.chdir(REPO)
for name, data in up.items():
    if name in CAN:
        with open(f"{REPO}/src/{name}", "wb") as f:
            f.write(data)
        print("đã thay", name)
    else:
        print("bỏ qua", name, "(không thuộc 5 file cần)")
thieu = [n for n in CAN if n not in up]
if thieu:
    raise SystemExit(f"THIẾU {thieu}: chạy lại ô này và chọn đủ 5 file")

## Bước 3b. Proxy và WARP
Ô 1 nạp danh sách proxy từ secret `ARXIV_PROXIES` (dạng `http://user:pass@ip:port` hoặc dạng Webshare
`ip:port:user:pass`, cách nhau bằng dấu phẩy). Proxy chỉ dùng cho lệnh gọi **API arXiv**; tải file không qua proxy
(để không tốn 1 GB của Webshare); Semantic Scholar không bao giờ qua proxy.

Ô 2 bật WARP: wgcf tạo tài khoản WARP miễn phí, wireproxy mở SOCKS5 ở `127.0.0.1:40000` (đúng cổng
`rate_limiter.py` dò lúc crawler khởi động). Ô tự kiểm tra `warp=on`; không kết nối được thì tự tắt và crawler
bỏ qua WARP. wireproxy có tắt giữa chừng thì 2 giây sau tự bật lại.

Đổi công tắc hay sửa secret thì chạy lại **cả 2 ô này và các ô của Bước 3c**.

In [ ]:
# ===== Công tắc =====
DUNG_PROXY = True   # xoay proxy Webshare (secret ARXIV_PROXIES) cho lệnh gọi API arXiv
DUNG_WARP = True    # bật WARP ở ô sau, dùng xen kẽ với mạng Colab khi tải file
# ====================
from google.colab import userdata

def chuan_hoa(p):
    p = p.strip()
    if p and "://" not in p and p.count(":") == 3:  # dạng Webshare ip:port:user:pass
        ip, port, user, pw = p.split(":")
        p = f"http://{user}:{pw}@{ip}:{port}"
    return p

os.environ.pop("ARXIV_PROXIES", None)
if os.path.isfile(f"{REPO}/src/proxies.txt"):  # rate_limiter.py tự đọc file này: không để nó lẫn vào
    os.remove(f"{REPO}/src/proxies.txt")
if not DUNG_PROXY:
    print("Proxy: tắt (API arXiv đi thẳng mạng Colab)")
else:
    try:
        raw = userdata.get("ARXIV_PROXIES")
        ds = [chuan_hoa(p) for p in raw.replace("\n", ",").replace(" ", ",").split(",") if p.strip()]
        os.environ["ARXIV_PROXIES"] = ",".join(ds)
        print(f"Proxy: đã nạp {len(ds)} proxy")
    except Exception as exc:
        print(f"Proxy: không nạp được ARXIV_PROXIES ({type(exc).__name__}): API arXiv đi thẳng mạng Colab. "
              "Kiểm tra secret và Notebook access.")

In [ ]:
# WARP: wgcf + wireproxy (WireGuard chạy trong userspace, không cần TUN). Phải chạy TRƯỚC Bước 4.
import socket, subprocess, time

WARP_DIR = "/content/warp"
WARP_PORT = 40000
WGCF_URL = "https://github.com/ViRb3/wgcf/releases/download/v2.3.1/wgcf_2.3.1_linux_amd64"
WIREPROXY_URL = "https://github.com/pufferffish/wireproxy/releases/download/v1.1.3/wireproxy_linux_amd64.tar.gz"
SOCKS = f"socks5h://127.0.0.1:{WARP_PORT}"

def cong_mo():
    try:
        with socket.create_connection(("127.0.0.1", WARP_PORT), timeout=0.5):
            return True
    except OSError:
        return False

def tat_warp():
    subprocess.run(["pkill", "-f", "chay_warp.sh"])     # vòng tự khởi động lại
    subprocess.run(["pkill", "-f", "wireproxy -s -c"])
    for _ in range(20):
        if not cong_mo():
            return
        time.sleep(0.25)

def trace():
    r = subprocess.run(["curl", "-s", "--max-time", "15", "-x", SOCKS,
                        "https://www.cloudflare.com/cdn-cgi/trace"], capture_output=True, text=True)
    return dict(line.split("=", 1) for line in r.stdout.splitlines() if "=" in line)

def chay(cmd, **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if r.returncode:
        dong = [l.strip() for l in (r.stderr or r.stdout).splitlines() if l.strip()] or ["?"]
        loi = [l for l in dong if any(k in l.lower() for k in ("error", "fail", "denied", "429", "lookup"))]
        raise RuntimeError(f"{os.path.basename(cmd[0])} {cmd[1] if len(cmd) > 1 else ''}: {(loi or dong)[0][-200:]}")

tat_warp()  # chạy lại ô này thì tắt bản cũ trước
if not DUNG_WARP:
    print("WARP: tắt (DUNG_WARP = False), tải file đi thẳng mạng Colab")
else:
    try:
        !pip install -q "requests[socks]"
        os.makedirs(WARP_DIR, exist_ok=True)
        wgcf, wp = f"{WARP_DIR}/wgcf", f"{WARP_DIR}/wireproxy"
        if not os.path.isfile(wgcf):
            chay(["curl", "-sSfL", "-o", wgcf, WGCF_URL])
            os.chmod(wgcf, 0o755)
        if not os.path.isfile(wp):
            chay(["curl", "-sSfL", "-o", f"{WARP_DIR}/wp.tar.gz", WIREPROXY_URL])
            chay(["tar", "-xzf", f"{WARP_DIR}/wp.tar.gz", "-C", WARP_DIR, "wireproxy"])
            os.chmod(wp, 0o755)
        if not os.path.isfile(f"{WARP_DIR}/wgcf-profile.conf"):
            chay([wgcf, "register", "--accept-tos"], cwd=WARP_DIR)
            chay([wgcf, "generate"], cwd=WARP_DIR)
        with open(f"{WARP_DIR}/wireproxy.conf", "w") as f:
            f.write(f"WGConfig = {WARP_DIR}/wgcf-profile.conf\n\n[Socks5]\nBindAddress = 127.0.0.1:{WARP_PORT}\n")
        with open(f"{WARP_DIR}/chay_warp.sh", "w") as f:  # wireproxy có tắt giữa chừng thì 2 s sau tự bật lại
            f.write(f"while true; do {wp} -s -c {WARP_DIR}/wireproxy.conf >> {WARP_DIR}/wireproxy.log 2>&1; sleep 2; done\n")
        subprocess.Popen(["bash", f"{WARP_DIR}/chay_warp.sh"], start_new_session=True,
                         stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        info = {}
        for _ in range(10):  # cổng mở chưa đủ: phải đi được ra Internet qua WARP
            time.sleep(2)
            info = trace() if cong_mo() else {}
            if info.get("warp") in ("on", "plus"):
                break
        if info.get("warp") in ("on", "plus"):
            print(f"WARP: đang chạy ở {SOCKS}, IP ra ngoài {info.get('ip')} ({info.get('colo')})")
        else:
            tat_warp()
            print("WARP: không kết nối được (Colab có thể chặn UDP 2408). Đã tắt; crawler tự bỏ qua WARP.")
    except Exception as exc:
        tat_warp()
        print(f"WARP: không bật được ({exc}). Crawler tự bỏ qua WARP, tải đi thẳng mạng Colab.")

## Bước 3c. Kiểm tra và nạp key
Ô 1 kiểm tra code và đường đi mà **không gửi gì tới arXiv**. Ô 2 thử từng đường bằng `api.ipify.org` và
`speed.cloudflare.com` (cũng không phải arXiv): proxy nào báo **HỎNG** thì xoá khỏi secret rồi chạy lại từ Bước 3b;
WARP chậm hơn mạng Colab nhiều thì đặt `DUNG_WARP = False`. Ô 3 nạp key Semantic Scholar.

In [ ]:
# Ô 1: kiểm tra code và đường đi (không gửi request nào): chạy trong tiến trình riêng, giống hệt lúc crawler khởi động
import subprocess, sys
KIEM_TRA = '''
import inspect, sys
sys.path.insert(0, "src")
import downloader, main, rate_limiter as rl
moi = "workers" in inspect.signature(downloader.run).parameters and hasattr(rl, "ARXIV_EXPORT")
lim = downloader.run.__kwdefaults__["limiter"]
print("downloader.py mới (tải 4 bài cùng lúc):", "CÓ" if moi else "KHÔNG")
print(f"Tải /src/ qua bộ giới hạn '{lim.name}': {lim.connections} kết nối, trung bình {lim.mean_interval:g} s/request")
print(f"API arXiv qua bộ giới hạn '{rl.ARXIV.name}': {rl.ARXIV.min_interval:g} s/request")
import re
an = lambda p: re.sub(r"//[^@/]*@", "//***@", p) if p else "mạng Colab"
print("Đường cho API arXiv:", len(rl.PROXIES), [an(p) for p in rl.PROXIES])
print("Đường cho tải file:", [an(p) for p in rl._UNLIMITED_LIST])
print("Có code tạm bỏ proxy hỏng (bản rate_limiter đã sửa):", "CÓ" if hasattr(rl, "_quarantine") else "KHÔNG")
sys.exit(0 if moi else 1)
'''
kq = subprocess.run([sys.executable, "-c", KIEM_TRA], capture_output=True, text=True, encoding="utf-8", cwd=REPO)
print(kq.stdout + kq.stderr[-3000:])
if kq.returncode:
    raise SystemExit("Chưa phải code mới: chạy lại ô 1 rồi ô 2 của bước này")

In [ ]:
# Thử từng đường (không gửi gì tới arXiv): IP ra ngoài và tốc độ tải
KIEM_DUONG = r"""
import re, sys, time
sys.path.insert(0, "src")
import requests, rate_limiter as rl

def an(p):
    return re.sub(r"//[^@/]*@", "//***@", p) if p else "mạng Colab"

print("IP ra ngoài của từng đường cho API arXiv:")
hong = []
for p in rl.PROXIES:
    try:
        ip = requests.get("https://api.ipify.org", timeout=15, proxies={"http": p, "https": p} if p else None).text
        print(f"   OK    {an(p):40} -> {ip}")
    except Exception as e:
        if p:
            hong.append(an(p))
        print(f"   HỎNG  {an(p):40} -> {type(e).__name__}")
print("\nTốc độ tải 10 MB qua các đường dùng để tải file:")
for p in rl._UNLIMITED_LIST:
    try:
        t = time.perf_counter()
        n = len(requests.get("https://speed.cloudflare.com/__down?bytes=10000000", timeout=60,
                             proxies={"http": p, "https": p} if p else None).content)
        print(f"   {an(p):40} {n / 1e6 / (time.perf_counter() - t):6.1f} MB/s")
    except Exception as e:
        print(f"   {an(p):40} lỗi {type(e).__name__}")
if hong:
    print(f"\n{len(hong)} proxy hỏng: {', '.join(hong)}")
    print("-> xoá khỏi secret ARXIV_PROXIES rồi chạy lại từ Bước 3b. Proxy hỏng không làm dừng crawler")
    print("   (nó bị bỏ qua 5 phút mỗi lần lỗi), nhưng mỗi lần lỗi tốn một request phải thử lại.")
"""
kq = subprocess.run([sys.executable, "-c", KIEM_DUONG], capture_output=True, text=True, encoding="utf-8", cwd=REPO)
print(kq.stdout + kq.stderr[-2000:])

In [ ]:
# Nạp key Semantic Scholar từ Secrets (không in key ra)
from google.colab import userdata
try:
    os.environ["S2_API_KEY"] = userdata.get("S2_API_KEY")
    print("Đã nạp S2_API_KEY")
except Exception as exc:
    print(f"Không nạp được S2_API_KEY ({type(exc).__name__}): vẫn chạy được, phần references chậm hơn")

## Bước 4. Chạy
Theo dõi các dòng `[download] ...`: dòng đầu phải có **`(4 papers at a time)`**.
Dòng `[proxy] … bỏ qua 5 phút` thỉnh thoảng xuất hiện là bình thường (proxy đó lỗi, request đi đường khác,
không tính là arXiv từ chối); cùng một proxy hiện liên tục thì xoá nó khỏi secret.
Runtime tự ngắt bất thường khi đang bật proxy/WARP: nhiều khả năng do Colab chặn, tắt 2 công tắc ở Bước 3b.
Muốn dừng giữa chừng: bấm ■ cạnh ô (mã 130); chạy lại ô này thì làm tiếp phần còn thiếu.
Dòng cuối có `Crawler return code: N`:

| N | Nghĩa |
|---|---|
| 0 | Xong → sang Bước 5 |
| 3 | arXiv từ chối liên tục 30 phút: để 1–2 giờ sau chạy lại ô này |
| 130 | Bạn bấm ■ |
| 2 | Có crawler khác đang ghi vào thư mục này: tắt tab/runtime kia trước |

In [ ]:
!python src/benchmark.py --data-dir "{THU}" --interval 2 --disk-interval 300 --heartbeat-seconds 60 -- python src/main.py --start {TU} --end {DEN} --data-dir "{THU}"

## Bước 5. Kiểm tra dữ liệu
Không có dòng `FAIL` là đạt. `PENDING` ở một phiên bản `not_found` nghĩa là arXiv không có file nguồn
cho phiên bản đó (404), không phải lỗi crawler.

In [ ]:
!python src/validate.py --data-dir "{THU}" --assigned-ids "{DS_ID}" --finished --progress-every 0 --output "{THU}/validation.csv"
import pandas as pd
v = pd.read_csv(f"{THU}/validation.csv", dtype=str)
khac = v[v.status != "PASS"]
print("Tất cả đều PASS" if khac.empty else "Các dòng không phải PASS:")
if not khac.empty:
    display(khac[["arxiv_id", "version", "check_name", "status", "error_message"]])

## Bước 6. Tốc độ và số lần arXiv từ chối

In [ ]:
!python src/group_benchmark.py m1="{THU}" --out "{THU}/tong_ket"
import glob, json
import pandas as pd
L = f"{THU}/logs"

def doc(ten):
    p = f"{L}/{ten}"
    return pd.read_csv(p) if os.path.isfile(p) and os.path.getsize(p) else pd.DataFrame()

print("\n== Thời gian từng bước (run_log.csv) ==")
rl_ = doc("run_log.csv")
if not rl_.empty:
    display(rl_[["stage", "seconds", "items", "note"]])

dl = doc("download_log.csv")
if not dl.empty:
    cuoi = dl.drop_duplicates(["arxiv_id", "version"], keep="last")
    print("Phiên bản theo trạng thái:", cuoi.status.value_counts().to_dict())
    print(f"Lần gửi request tải (kể cả thử lại): {pd.to_numeric(dl.attempts, errors='coerce').sum():.0f}")
    tai = rl_[rl_.stage == "download"] if not rl_.empty else rl_
    if not tai.empty and len(cuoi):
        print(f"Bước tải: {tai.seconds.sum():.0f} s cho {len(cuoi)} phiên bản = {tai.seconds.sum() / len(cuoi):.2f} s/phiên bản")

th = doc("arxiv_throttle_log.csv")
if th.empty:
    print("\narXiv không từ chối lần nào (arxiv_throttle_log.csv trống)")
else:
    print("\narXiv từ chối/nghỉ:", th.event.value_counts().to_dict(),
          f"| tổng thời gian nghỉ {pd.to_numeric(th.total_paused_seconds, errors='coerce').max():.0f} s")
    display(th.tail(10))

pl = doc("pass_log.csv")
if not pl.empty:
    print("\n== Các vòng (pass_log.csv) ==")
    display(pl)

tom = sorted(glob.glob(f"{L}/performance_summary_*.json"))
if tom:
    s = json.load(open(tom[-1], encoding="utf-8"))
    wall = s.get("wall_seconds") or 0
    print(f"\nLần chạy cuối: {wall:.0f} s cho {SO_ID} ID, mã thoát {s.get('return_code')}, "
          f"RAM đỉnh {(s.get('ram_sampled_peak_bytes') or 0) / 2**20:.0f} MB")
    if wall and s.get("started_with_fresh_data_dir"):
        print(f"Ước chừng cho 5000 ID (nhân tuyến tính, chỉ tham khảo): {wall / SO_ID * 5000 / 3600:.1f} giờ")

## Sau khi chạy thử ổn
- Đạt khi: Bước 4 ra `Crawler return code: 0`, có `(4 papers at a time)`, Bước 5 không có `FAIL`.
- Khi đó push 5 file trong `src`, `README.md` và `.gitignore` (**không** push `proxies.txt`), và thêm dòng `PySocks==1.7.1` vào `src/requirements.txt` (WARP cần).
  Sau khi push thì không cần ô tải 5 file nữa; chạy chính thức bằng notebook `chay-crawler-colab.ipynb` vào `01_crawl/m1`.
- Thư mục `02_trial/m1_...` chỉ là dữ liệu thử, xoá được khi không cần.